# And

Bloqs for doing "AND" logical operations.

The behavior is modified by the 'control variable' attributes. A traditional value of '1'
means that a bit value of '1' is logical true for the and operation. A control value of
'0' means that a bit value of '0' is the logical true.

The `Toffoli` bloq is similar to the `And` bloq. Toffoli will flip the target bit according
to the and of its control registers. `And` will output the result into a fresh register.

In [ ]:
from qualtran import Bloq, CompositeBloq, BloqBuilder, Signature, Register
from qualtran import QBit, QInt, QUInt, QAny
from qualtran.drawing import show_bloq, show_call_graph, show_counts_sigma
from typing import *
import numpy as np
import sympy
import cirq

## `And`
A two-bit 'and' operation optimized for T-count.

#### Parameters
 - `cv1`: Whether the first bit is a positive control.
 - `cv2`: Whether the second bit is a positive control. 

#### Registers
 - `ctrl`: A two-bit control register.
 - `target [right]`: The output bit. 

#### References
 - [Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity](https://arxiv.org/abs/1805.03662).     Babbush et al. 2018. Section III.A. and Fig. 4.
 - [Verifying Measurement Based Uncomputation](https://algassert.com/post/1903). Gidney, C. 2019.


In [ ]:
from qualtran.bloqs.mcmt import And

### Example Instances

In [ ]:
and_bloq = And()

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([and_bloq],
           ['`and_bloq`'])

### Call Graph

In [ ]:
from qualtran.resource_counting.generalizers import ignore_split_join
and_bloq_g, and_bloq_sigma = and_bloq.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(and_bloq_g)
show_counts_sigma(and_bloq_sigma)

### Clifford + T implementation

`And` will be considered an Atomic (non-decomposable) bloq within the Qualtran standard library. An additional method is provided to assist in fully compiling to a Clifford+T gateset.

In [ ]:
and_bloq.to_clifford_t_circuit()

In [ ]:
and_bloq.adjoint().to_clifford_t_circuit()

## MBUC tensor

In [ ]:
from qualtran.bloqs.basic_gates import PlusState, ZeroState, Hadamard, XGate, ZGate
from qualtran.bloqs.basic_gates.x_basis import MeasX
from qualtran.bloqs.basic_gates.z_basis import MeasZ
from qualtran.bloqs.basic_gates.discard import Discard, DiscardQ
from qualtran.bloqs.bookkeeping import Cast

from qualtran import BloqBuilder, CtrlSpec, QBit, CBit, Register, Side
from qualtran.drawing import show_bloq
from qualtran.simulation.tensor._quimb import cbloq_to_superquimb, cbloq_to_quimb

bb = BloqBuilder()
q = bb.add_register(Register("q", QBit(), side=Side.LEFT))
bb.add_register(Register("c", CBit(), side=Side.RIGHT))

op = ZGate()
op = XGate()

meas_space = bb.add(ZeroState())
meas_space = bb.add(Hadamard(), q=meas_space)

_, add_ctrled = op.get_ctrl_system(CtrlSpec())
(meas_space,), (q,) = add_ctrled(bb, ctrl_soqs=[meas_space], in_soqs={'q': q})

meas_space = bb.add(Hadamard(), q=meas_space)
meas_result = bb.add(Cast(QBit(), CBit()), reg=meas_space)
bb.add(DiscardQ(), x=q)
meas_cbloq = bb.finalize(c = meas_result)
show_bloq(meas_cbloq)

In [ ]:
from qualtran import Side

from qualtran.bloqs.basic_gates.x_basis import MeasX, PlusState
from qualtran.bloqs.basic_gates.z_basis import MeasZ, ZeroState
from qualtran.bloqs.basic_gates import CZ, Hadamard, CNOT
from qualtran.bloqs.basic_gates.discard import Discard

from qualtran import CtrlSpec, Controlled, CBit

bb = BloqBuilder()
q1 = bb.add_register('q1', 1)
q2 = bb.add_register('q2', 1)
trg = bb.add_register(Register('trg', QBit(), side=Side.LEFT))

#ctrg, = bb.add_from(meas_cbloq, q=trg)
ctrg = bb.add(MeasX(), q=trg)
#trg = bb.add(Hadamard(), q=trg)
#ctrg = bb.add(MeasZ(), q=trg)

classicall_controlled_cz = CZ().controlled(CtrlSpec(qdtypes=(CBit(),)))

ctrg, q1, q2 = bb.add(
    classicall_controlled_cz,
    **{classicall_controlled_cz.ctrl_reg_names[0]: ctrg,
       'q1': q1,
       'q2': q2
      }
)
bb.add(Discard(), x=ctrg)

#ccz, add_ccz = CZ().get_ctrl_system(CtrlSpec())
#add_ccz(bb, ctrl_soqs=[ctrg], in_soqs={'q1': q1, 'q2': q2})


cbloq = bb.finalize(q1=q1, q2=q2)
show_bloq(cbloq)

In [ ]:
classicall_controlled_cz.tensor_contract()

In [ ]:
bb = BloqBuilder()
q = bb.add_register('q', 1)
q = bb.add(Hadamard(), q=q)

m = bb.add(ZeroState())
q, m = bb.add(CNOT(), ctrl=q, target=m)


meascbloq = bb.finalize(q=q, m=m)
show_bloq(meascbloq)

In [ ]:
meascbloq.signature

In [ ]:
meascbloq.tensor_contract()

In [ ]:
#meast = meastn.contract()
#meast

In [ ]:
#meast.data.transpose(1,2,0).reshape((2*2, 2))

In [ ]:
from qualtran.bloqs.mcmt import And

bb = BloqBuilder()
q1 = bb.add_register('q1', 1)
q2 = bb.add_register('q2', 1)
(q1, q2), trg = bb.add(And(), ctrl=[q1,q2])

# Our real circuit uncomputes (AND + NAND), so if we add
# a full X here ... it should still work (?)
# trg = bb.add(XGate(), q=trg)

q1, q2 = bb.add_from(cbloq, q1=q1, q2=q2, trg=trg)
cunc = bb.finalize(q1=q1, q2=q2)
show_bloq(cunc)

In [ ]:
from qualtran.simulation.tensor._quimb import cbloq_to_superquimb
tn = cbloq_to_superquimb(cunc, friendly_indices=True)
tn.draw(color=['MeasX', 'And'])

In [ ]:
tn.contract()

In [ ]:
np.where(np.abs(tn.to_dense(
    ['q1_0rf', 'q2_0rf', 'q1_0rb', 'q2_0rb'],
    ['q1_0lf', 'q2_0lf', 'q1_0lb', 'q2_0lb']
) - 1.0) < 1e-8)

In [ ]:
tn.to_dense(
    ['q1_0rf', 'q2_0rf', 'q1_0rb', 'q2_0rb'],
    ['q1_0lf', 'q2_0lf', 'q1_0lb', 'q2_0lb']
)

In [ ]:
import quimb.tensor as qtn
tnf, tnb = tn.split(method='svd',
                     left_inds = ['q2_0lf', 'q1_0lf', 'q2_0rf', 'q1_0rf'],
                     absorb='both', bond_ind='j')
display(tnf)
display(tnf.data[:,:,:,:,0].reshape(4,4).round(4))

In [ ]:
tnsplit = tnf.isel({'j': 0})
tnsplit

In [ ]:
tnsplit.data

### Superop

In [ ]:
tn = cbloq_to_superquimb(cbloq, friendly_indices=True)
tn.draw(color=['MeasX', 'And'])

In [ ]:
tn.outer_inds()

In [ ]:
import quimb.tensor as qtn
tnf, tnb = tn.split(method='svd',
                     left_inds = ['q2_0lf', 'q1_0lf', 'q2_0rf', 'q1_0rf', 'trg_0lf'],
                     absorb='both', bond_ind='j')
display(tnf)
display(tnb)

In [ ]:

print("Selecting kraus index j=0...\n")
print("  q1q2 trg |  And? ")
print("  ---------+-------")
ttt = tnf.isel({'j': 0})
for indvals in np.array(np.where(np.abs(ttt.data) > 1e-4)).T:
    q2l, q1l, q2r, q1r, trg = indvals
    assert q1l == q1r
    assert q2l == q2r
    trg_should_be = int(q1l==1 and q2l==1)
    print(f"   {q1l}{q2l}   {trg}  |  {trg==trg_should_be}")

print('\n')
print("Selecting kraus index j=1...\n")
print("  q1q2 trg |  And? ")
print("  ---------+-------")
ttt = tnf.isel({'j': 1})
for indvals in np.array(np.where(np.abs(ttt.data) > 1e-4)).T:
    q2l, q1l, q2r, q1r, trg = indvals
    assert q1l == q1r
    assert q2l == q2r
    trg_should_be = int(q1l==1 and q2l==1)
    print(f"   {q1l}{q2l}   {trg}  |  {trg==trg_should_be}")

## `MultiAnd`
A many-bit (multi-control) 'and' operation.

#### Parameters
 - `cvs`: A tuple of control variable settings. Each entry specifies whether that control line is a "positive" control (`cv[i]=1`) or a "negative" control `0`. If a HasLength object is passed, assumes the control values to be all 1's.  

#### Registers
 - `ctrl`: An n-bit control register.
 - `junk [right]`: An `n-2` bit junk register to be cleaned up by the inverse operation.
 - `target [right]`: The output bit.


In [ ]:
from qualtran.bloqs.mcmt import MultiAnd

### Example Instances

In [ ]:
multi_and = MultiAnd(cvs=(1, 0, 1, 0, 1, 0))

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs
show_bloqs([multi_and],
           ['`multi_and`'])

### Call Graph

In [ ]:
from qualtran.resource_counting.generalizers import ignore_split_join
multi_and_g, multi_and_sigma = multi_and.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(multi_and_g)
show_counts_sigma(multi_and_sigma)

## Additional Demos

### Testing with states and effects

We can use `ZeroState` and its friends to test the truth table on this classical logic.

In [ ]:
from qualtran.bloqs.basic_gates import OneEffect, OneState, ZeroEffect, ZeroState

state = [ZeroState(), OneState()]
eff = [ZeroEffect(), OneEffect()]

# Experiment with changing the following:
cvs = (1, 1, 1)
ctrl_string = (1, 1, 1)


bb = BloqBuilder()
ctrl_qs = [bb.add(state[c]) for c in ctrl_string]
ctrl_qs, junk, res = bb.add_from(MultiAnd(cvs), ctrl=ctrl_qs)
for c, q in zip(ctrl_string, ctrl_qs):
    bb.add(eff[c], q=q)

cbloq = bb.finalize(junk=junk, res=res)
show_bloq(cbloq)

In [ ]:
# Our tensor network now just has the result index and a junk index.
# We use `np.where` to find non-zero entries into this.
# In fact -- the second index corresponding to `res` is the bit output
vec = cbloq.tensor_contract()
junk_i, res_i = np.where(vec.reshape((2, 2)))
res_i

In [ ]:
# The truthiness of the non-zero res index should match the desired logical function.
should_be = np.all(ctrl_string == cvs)
should_be

### Classical Simulation

The `And` gate is classical logic, so we can simulate it on discrete bitstrings.

In [ ]:
ctrl, out = And().call_classically(ctrl=np.array([1, 1]))
out

In [ ]:
ctrl = np.array([1,1,1,1])
ctrl, junk, out = MultiAnd((1,1,1,1)).call_classically(ctrl=ctrl)
out

In [ ]:
from qualtran.drawing import ClassicalSimGraphDrawer

ClassicalSimGraphDrawer(
    MultiAnd((1,1,1,1)).decompose_bloq(), 
    vals=dict(ctrl=[1,1,0,1])
).get_svg()